In [1]:
# リカレントニューラルネットワーク（RNN）
# 確率と言語モデル
# word2vecを確率の視点から眺める
# 言語モデル
# CBOWモデルを言語モデルに？

In [ ]:
# RNNとは
# フィードフォワードとは
# 循環するニューラルネットワーク
# ループの展開
# Backpropagation Through Time
# Truncated BPTT
# Truncated BPTTのミニバッチ学習


In [2]:
# RNNとは
# 循環するニューラルネットワーク
# ループの展開
# Backpropagation Through Time
# Truncated BPTT
# Truncated BPTTのミニバッチ学習

In [ ]:
# RNNの実装
# RNNレイヤの実装
import numpy as np

class RNN:
    # Wx: 入力x用の重み  Wh: 前の隠れ状態h用の重み  b: バイアス
    def __init__(self, Wx, Wh, b):
        # 外部から受け取った重みをリストにまとめる
        # Optimizerで一気に更新できるようにするため
        # Optimizer = grads(勾配)を一気に更新する
        self.params = [Wx, Wh, b] 
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)] # 重みと全く同じ形をした「0」の箱
        self.cache = None  # forwardの最後でデータを一時保存するための箱

    def forward(self, x, h_prev):
        Wx, Wh, b = self.params
        # h_prev, h_next = 今何を覚えているかのメモ(記憶そのもの)
        # Wh = その記憶を次のステップでどう使いこなすかを示す重み
        # h_prevに重みを掛けることで、過去の情報（h_prev）をどう解釈するのかを学習する
        t = h_prev @ Wh + x @ Wx + b
        # なぜ「t」にnp.tanhを通すのか
        # tのままだと、足し算や掛け算を繰り返すうちに、数値がどんどん巨大になってしまう（勾配爆発）
        # 'np.tanh'を通すことで最大1.0、最小-1.0の間に保てる
        h_next = np.tanh(t)

        self.cache = (x, h_prev, h_next)  # backwardで使うためにメモを残しておく
        return h_next

    def backward(self, dh_next):
        Wx, Wh, b = self.params
        x, h_prev, h_next = self.cache  # forwardで保存したメモを取り出す

        dt = dh_next * (1 - h_next ** 2)
        db = np.sum(dt, axis=0)
        dWh = h_prev.T @ dt
        dh_prev = dt @ Wh.T
        dWx = x.T @ dt
        dx = dt @ Wx.T

        self.grads[0][...] = dWx
        self.grads[1][...] = dWh
        self.grads[2][...] = db

        return dx, dh_prev

In [ ]:
# Time RNNレイヤの実装
class TimeRNN:
    def __init__(self, Wx, Wh, b, stateful=False):
        # 重みとバイアス、勾配をリストにまとめる
        self.params =[Wx, Wh, b] 
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        self.layers = None
        
        # self.h：隠れ状態を保存しておく場所
        # self.dh：逆伝播時に、前のブロックから引き継いだ反省内容を
        # 一時的に保存しておく場所
        self.h, self.dh = None, None
        # stateful = 記憶を維持するかどうかのスイッチ
        # True：self.hを捨てない（Truncated BPTTの基本）
        # False：self.hをリセットする
        self.stateful = stateful

    # 外側からスタート位置を指定する
    def set_state(self, h):
        self.h = h
    # 記憶を一旦リセットする
    def reset_state(self):
        self.h = None

    
    def forward(self, xs):
        # 重みを取り出す
        Wx, Wh, b = self.params
        # xs：データの形を把握する
        # N：Batch Size（同時に処理するサイズ）
        # T：Time Steps（時間の長さ）
        # D：Input Dimension（1文字を何次元のベクトルで表すか）
        N, T, D = xs.shape
        # 重みWxの形を見て、記憶のサイズを把握
        # D：入力の次元（xs.shapeから受け取ったものと同じ）
        # H：Hidden Size（隠れ状態を何次元にするか）
        D, H = Wx.shape

        # レイヤをまとめるための箱を用意
        self.layers = []
        # np.empty：指定した形のメモリ領域を確保
        # → 別の計算で使われていたゴミデータが残っている
        # np.zerosではなく、np.emptyを使うのは、
        # 値をすぐに上書きする場合、emptyの方が効率が良いから
        hs = np.empty((N, T, H), dtype="f")

        # self.hの初期化
        # 隠れ状態を引き継ぐかリセットするかをif文で判断
        if not self.stateful or self.h is None:
            self.h = np.zeros((N, H), dtype="f")

            # 時系列を回すためのfor文
            for t in range(T):
                # RNNレイヤを文字数分量産する（設計図は全部同じ）
                layer = RNN(*self.params)
                # self.h：保存されてる隠れ状態
                # RNNレイヤのインスタンスが持ってるforwardメソッドを呼び出す
                # layer.forward(今の文字, これまでの記憶)
                self.h = layer.forward(xs[:, t, :], self.h)
                # 次のレイヤに通して予測するために、selfh.hをhsに代入
                hs[:, t, :] = self.h
                # 逆伝播用にlayerをself.layersに登録
                self.layers.append(layer)

            return hs

    def backward(self, dhs):
        # 重みを取りだす
        Wx, Wh, b = self.params
        # N：Batch Size（同時に処理するサイズ）
        # T：Time Steps（時間の長さ）
        # H：Hidden Size（記憶の次元数）
        N, T, H = dhs.shape
        # Dの次元を把握するために、重みWxの形を確認する
        D, H = Wx.shape

        # np.empty：指定した形のメモリ領域を確保
        # dxs：勾配
        dxs = np.empty((N, T, D), dtype="f")
        # dh：記憶に対する勾配
        dh = 0
        grads = [0, 0, 0]
        for t in reversed(range(T)):
            layer = self.layers[t]
            # 
            dx, dh = layer.backward(dhs[:, t, :] + dh)
            dxs[:, t, :] = dx

            for i, grad in enumerate(layer.grads):
                grads[i] += grads

            for i, grad in enumerate(grads):
                self.grads[i][...] = grad
            self.dh = dh

            return dxs

In [7]:
# 時系列データを扱うレイヤの実装
# RNNLMの全体図
# Timeレイヤの実装
# RNNLMの学習と評価

In [ ]:
# RNNLMの実装
import sys
sys.path.append("..")
import numpy as np
from common.time_layers import *

class SimpleRnnlm:
    def __init__(self, vocab_size, wordvec_size, hidden_size):
        # V：語彙数　D：単語を何次元のベクトルで表現するか　H：RNNの隠れ状態
        V, D, H = vocab_size, wordvec_size, hidden_size
        rn = np.random.randn

        # 重みの初期化
        # 「V個の単語」それぞれに「D次元の意味」を割り当てる
        embed_W = (rn(V, D) / 100).astype("f")
        # 「今の入力D」を「記憶H」に変換するための重み
        rnn_Wx = (rn(D, H) / np.sqrt(D)).astype("f")
        # 「これまでの記憶H」を「次の記憶H」に引き継ぐための重み
        rnn_Wh = (rn(H, H) / np.sqrt(H)).astype("f")
        rnn_b = np.zeros(H).astype("f")
        # 「今の記憶H」から「次に来る単語の候補V個」を絞りだすための重み
        affine_W = (rn(H, V) / np.sqrt(H)).astype("f")
        affine_b = np.zeros(V).astype("f")

        # レイヤの生成
        self.layers = [
            TimeEmbedding(embed_W),
            TimeRNN(rnn_Wx, rnn_Wh, rnn_b, stateful=True),
            TimeAffine(affine_W, affine_b)
        ]
        # スコア出し
        self.loss_layer = TimeSoftmaxWithLoss()
        # RNNを呼び出すためのショートカットコード
        self.rnn_layer = self.layers[1]

        # 全ての重みと勾配をリストにまとめる
        self.params, self.grads = [], []
        for layer in self.layers:
            self.params += layer.params
            self.grads += layer.grads
     
    def forward(self, xs, ts):
        # このfor文で、入力値xsがどんどん上書きされて変化していく
        # TimeEmbedding：xsは「ベクトルの塊」に変化
        # TimeRNN：xsは「隠れ状態の塊」に変化
        # TimeAffine：xsは「次に来る単語の予測スコア」に変化
        for layer in self.layers:
            xs = layer.forward(xs)
        # xs：今作った「予測スコア」
        # ts：実際の正解データ
        # 誤差を計算する
        loss = self.loss_layer.forward(xs, ts)
        return loss

    def backward(self, dout=1):
        # loss_layerから勾配を受け取る
        dout = self.loss_layer.backward(dout)
        # レイヤを後ろから回る
        for layer in reversed(self.layers):
            # 予測のズレ → 記憶のズレ → 入力単語のズレへと変化する
            dout = layer.backward(dout)
        return dout

    # TimeRNNレイヤに向けて、隠れ状態のリセットを命令する
    def reset_state(self):
        self.rnn_layer.reset_state()


In [ ]:
# 言語モデルの評価
# パープレキシティ


In [ ]:
# RNNLMの学習コード
import sys
sys.path.append("..")
import matplotlib.pyplot as plt
from common.optimizer import SGD
from dataset import ptb
from simple_rnnlm import SimpleRnnlm

# ハイパーパラメータの設定
batch_size = 10  # 10個ずつ学習を進める
wordvec_size = 100  # 単語ベクトルの要素数
hidden_size = 100  # RNNの隠れ状態ベクトルの要素数
time_size = 5  # Truncated BPTTの展開する時間サイズ
lr = 0.1  # 学習率
max_epoch = 100

# 学習データの読み込み（データセットを小さくする）
corpus, word_to_id, id_to_word = ptb.load_data("train")
corpus_size = 1000  # 巨大なPTBデータセットから先頭の1000単語を抜き出す
corpus = corpus[:corpus_size]  
vocab_size = int(max(corpus) + 1)

# 「今の単語に対して、次に来る単語を正解にする」というセットを作成
xs = corpus[:-1]  # 最初から、最後の1つ手前まで(入力)
ts = corpus[1:]  # 2番目から最後まで（正解）
data_size = len(xs)  # xsが何組あるか
print("corpus size: %d, vocabulary size: %d" % (corpus_size, vocab_size))

# 学習時に使用する変数
# batch_size * time_size：batch_size分で、それぞれ5文字ずつ読むから、1回で50単語計算
# 全体をデータ量（50）で割る
max_iters = data_size // (batch_size * time_size)
time_idx = 0  # 文章の何文字目を読んでいるかの印（学習が進む度に変化）
total_loss = 0  # 損失を足すための箱
loss_count = 0  # 平均を出すためのカウント  ➡  total_loss, loss_count： パープレキシティの材料
ppl_list = []  # 計算したパープレキシティを順番に保存しておくためのリスト

# モデルの生成
# vocab_size：語彙数
# wordvec_size：単語のベクトルの次元数
# hidden_size：記憶の次元数
# SimpleRnnlmのforward, backwardが実行される
model = SimpleRnnlm(vocab_size, wordvec_size, hidden_size)
# SGD：確率的勾配降下法
# 0.1のペースでコツコツと重みを書き換える
optimizer = SGD(lr)

# ミニバッチの各サンプルの読み込み開始位置を計算
# corpusを10個のエリアに均等に分けるために、
# 1人あたり何文字分担当するのかを計算
jump = (corpus_size - 1) // batch_size
# 10個のエリアそれぞれの開始地点をリストにする
# 1人目：0 * 100 = 0
# 2人目：1 * 100 = 100
# 3人目：2 * 100 = 200 ・・・
offsets = [i * jump for i in range(batch_size)]

# データセットを何回繰り返すか
for epoch in range(max_epoch):
    # 1エポックの中で「予測 → 反省 → 重み更新」を何回繰り返すのか
    for iter in range(max_iters):
        # ミニバッチの取得
        # 
        batch_x = np.empty((batch_size, time_size), dtype="i")
        batch_t = np.empty((batch_size, time_size), dtype="i")
        for t in range(time_size):
            for i, offset in enumerate(offsets):
                batch_x[i, t] = xs[(offset + time_idx) % data_size]
                batch_t[i, t] = ts[(offset + time_idx) % data_size]
            time_idx += 1
        
        # 勾配を求め、パラメータを更新
        loss = model.forward(batch_x, batch_t)
        model.backward()
        optimizer.update(model.params, model.grads)
        total_loss += loss
        loss_count += 1

# エポックごとにパープレキシティの評価
ppl = np.exp(total_loss / loss_count)
print("| epoch %d | perplexity %.2f" % (epoch+1, ppl))
ppl_list.append(float(ppl))
total_loss, loss_count = 0, 0
        